# Cell Trajectory and Shape Visualization

This notebook generates visualizations of cell paths and shapes over time by overlaying 
cell boundaries extracted from the simulation frames onto the maze background.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import cv2
from typing import List, Tuple

REPO_ROOT = Path(os.getcwd()).parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Imports successful.")

In [ ]:
# Plotting Constants
TITLE_FONTSIZE = 16
SUBTITLE_FONTSIZE = 14
AXIS_LABEL_FONTSIZE = 12
TICK_LABEL_FONTSIZE = 10
LEGEND_FONTSIZE = 10


In [ ]:
DATA_DIR = REPO_ROOT / "output"
SIMULATIONS_DIR = REPO_ROOT / "data" / "sample_simulations"

# Load data
summary_df = pd.read_pickle(DATA_DIR / "collision_analysis_summary.pkl")
detailed_df = pd.read_pickle(DATA_DIR / "collision_analysis_detailed.pkl")

print(f"Loaded {len(summary_df)} simulations")


In [ ]:
def extract_cell_shape_data(simulation_id: str, 
                           time_interval: int = 1, 
                           save_path: str = None) -> pd.DataFrame:
    """
    Extract detailed cell shape data from all frames of a simulation.
    
    Args:
        simulation_id: ID of the simulation to process
        time_interval: Process every Nth frame (default=1 means every frame)
        save_path: Optional path to save the resulting DataFrame
        
    Returns:
        DataFrame with columns: simulation_id, timestep, x, y, frame_index
        Each row represents a single pixel of the cell at a specific timestep
    """
    # Find the simulation folder path from summary_df
    sim_row = summary_df[summary_df['simulation_id'] == str(simulation_id)]
    if sim_row.empty:
        raise ValueError(f"Simulation ID {simulation_id} not found in summary data")
        
    folder_path = sim_row['folder_path'].iloc[0]
    # Fix the folder path if needed
    folder_path = fix_folder_path(folder_path)
    
    model_type = sim_row['model_type'].iloc[0]
    print(f"Extracting cell shape data for simulation {simulation_id} ({model_type}) from {folder_path}")
    
    # Get the PNG files for this simulation
    png_files = find_simulation_pngs(folder_path)
    
    if len(png_files) == 0:
        raise ValueError(f"No PNG files found for simulation {simulation_id}")
        
    # Process every nth frame based on time_interval
    cell_data = []
    total_frames = len(png_files)
    
    # Progress tracking
    print(f"Processing {len(png_files[::time_interval))} frames out of {total_frames} total frames")
    
    for i, (timestep, filepath) in enumerate(png_files[::time_interval)):
        if i % 10 == 0:
            print(f"  Processing frame {i} (timestep {timestep})")
            
        # Load image and detect black pixels (the cell)
        try:
            image = cv2.imread(filepath)
            if image is None:
                print(f"  Warning: Could not load image {filepath}, skipping")
                continue
                
            # Create binary mask of cell (black) pixels
            black_mask = detect_black_pixels(image, tolerance=10)
            
            # Get coordinates of all cell pixels
            cell_pixel_coords = np.where(black_mask == 255)
            
            # Skip frames where no cell is detected
            if len(cell_pixel_coords[0)) == 0:
                continue
                
            # Convert to rows for DataFrame with image coordinates
            h, w = black_mask.shape
            frame_rows = []
            
            # Take a sample of pixels if there are too many (for performance)
            sampling_rate = max(1, len(cell_pixel_coords[0)) // 1000)
            
            for y_idx, x_idx in zip(cell_pixel_coords[0][::sampling_rate], 
                                    cell_pixel_coords[1][::sampling_rate)):
                # Convert to coordinate system where (0,0) is bottom-left
                y_coord = h - y_idx  # Flip Y coordinate
                x_coord = x_idx
                
                frame_rows.append({
                    'simulation_id': simulation_id,
                    'model_type': model_type,
                    'timestep': timestep,
                    'frame_index': i,
                    'x': x_coord,
                    'y': y_coord,
                })
                
            cell_data.extend(frame_rows)
            
        except Exception as e:
            print(f"  Error processing frame {timestep}: {e}")
    
    # Convert to DataFrame
    if not cell_data:
        raise ValueError(f"No valid cell data extracted from simulation {simulation_id}")
        
    cell_df = pd.DataFrame(cell_data)
    
    print(f"Extracted {len(cell_df)} cell pixel positions across {cell_df['timestep'].nunique()} timesteps")
    
    # Save to file if requested
    if save_path:
        cell_df.to_pickle(save_path)
        print(f"Cell shape data saved to {save_path}")
        
    return cell_df

def plot_cell_path_evolution(cell_df: pd.DataFrame, 
                             background_image_path: str = None,
                             output_path: str = None,
                             time_bins: int = 10,
                             alpha_gradient: bool = True):
    """
    Plot the full path evolution of a cell using its pixel data.
    
    Args:
        cell_df: DataFrame with cell pixel positions from extract_cell_shape_data()
        background_image_path: Optional path to a background maze image
        output_path: Optional path to save the figure
        time_bins: Number of color bins for time progression (fewer = clearer progression)
        alpha_gradient: If True, newer positions are more opaque
        
    Returns:
        matplotlib figure and axes
    """
    if cell_df.empty:
        raise ValueError("Empty cell data provided")
    
    # Setup the figure
    fig, ax = plt.subplots(figsize=(16, 12))
    
    # Get simulation info for title
    sim_id = cell_df['simulation_id'].iloc[0]
    model_type = cell_df['model_type'].iloc[0]
    
    # Load background image if provided
    if background_image_path:
        if Path(background_image_path).exists():
            bg_img = cv2.imread(background_image_path)
            bg_img_rgb = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)
            ax.imshow(bg_img_rgb)
        else:
            print(f"Warning: Background image not found: {background_image_path}")
            
            # Try to get a background from the simulation's first frame
            try:
                sim_row = summary_df[summary_df['simulation_id'] == str(sim_id)]
                if not sim_row.empty:
                    folder_path = sim_row['folder_path'].iloc[0]
                    png_files = find_simulation_pngs(folder_path)
                    if png_files:
                        # Use the first frame as background
                        bg_img = cv2.imread(png_files[0][1))
                        bg_img_rgb = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)
                        ax.imshow(bg_img_rgb)
                        print(f"Using simulation's first frame as background")
            except Exception as e:
                print(f"Could not load simulation background: {e}")
    
    # Create time bins for coloring
    min_timestep = cell_df['timestep'].min()
    max_timestep = cell_df['timestep'].max()
    timestep_range = max_timestep - min_timestep
    
    # Create a colormap for time progression
    colormap = plt.cm.viridis
    
    # Group by timestep to reduce plotting overhead
    grouped_df = cell_df.groupby('timestep')
    
    # Get a list of all unique timesteps
    timesteps = sorted(cell_df['timestep'].unique())
    
    # Create time bins for better visualization
    time_bin_edges = np.linspace(0, len(timesteps), time_bins + 1, dtype=int)
    binned_timesteps = [timesteps[i:j] for i, j in zip(time_bin_edges[:-1], time_bin_edges[1:))]
    
    # Plot each time bin with a different color
    for bin_idx, bin_timesteps in enumerate(binned_timesteps):
        # Get normalized color value for this bin (0 to 1)
        color_val = bin_idx / (len(binned_timesteps) - 1)
        
        # Set alpha (transparency) - earlier bins are more transparent if alpha_gradient is True
        alpha = 0.3 + 0.7 * (bin_idx / (len(binned_timesteps) - 1)) if alpha_gradient else 0.7
        
        # Collect all points in this time bin
        bin_points = cell_df[cell_df['timestep'].isin(bin_timesteps)]
        
        if not bin_points.empty:
            # Plot this time bin
            ax.scatter(bin_points['x'], bin_points['y'], 
                      color=colormap(color_val),
                      s=1.5,  # Small point size for detailed visualization
                      alpha=alpha,
                      label=f"Time {bin_idx+1}/{len(binned_timesteps)}")
    
    # Set plot title and labels
    ax.set_title(f"Cell Path Evolution - Model: {model_type}, Sim: {sim_id}", 
                fontsize=TITLE_FONTSIZE)
    ax.set_xlabel("X Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel("Y Position", fontsize=AXIS_LABEL_FONTSIZE)
    
    # Create custom legend with time progression
    handles = [plt.Line2D([0], [0], marker='o', color='w', 
                         markerfacecolor=colormap(i/(time_bins-1)), 
                         markersize=10, label=f"Time {i+1}/{time_bins}")
              for i in range(time_bins)]
    ax.legend(handles=handles, title="Time Progression", 
             loc='upper right', fontsize=TICK_LABEL_FONTSIZE)
    
    # Save the figure if output path is specified
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"Path evolution plot saved to {output_path}")
    
    plt.tight_layout()
    return fig, ax

def create_cell_shape_animation(cell_df: pd.DataFrame, 
                              background_image_path: str = None, 
                              output_path: str = None,
                              fps: int = 10,
                              frame_interval: int = 1):
    """
    Create an animation of the cell moving through the maze.
    
    Args:
        cell_df: DataFrame with cell pixel positions from extract_cell_shape_data()
        background_image_path: Optional path to a background maze image
        output_path: Path to save the animation (mp4 or gif)
        fps: Frames per second for the animation
        frame_interval: Use every Nth frame (higher = faster animation)
        
    Returns:
        matplotlib animation object
    """
    if cell_df.empty:
        raise ValueError("Empty cell data provided")
        
    # Setup the figure
    fig, ax = plt.subplots(figsize=(12, 9))
    
    # Get simulation info for title
    sim_id = cell_df['simulation_id'].iloc[0]
    model_type = cell_df['model_type'].iloc[0]
    
    # Load background image if provided
    if background_image_path:
        if Path(background_image_path).exists():
            bg_img = cv2.imread(background_image_path)
            bg_img_rgb = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)
            ax.imshow(bg_img_rgb)
            
            # Store image dimensions for proper axes limits
            img_height, img_width = bg_img.shape[:2]
        else:
            print(f"Warning: Background image not found: {background_image_path}")
            img_height = max(cell_df['y')) + 50
            img_width = max(cell_df['x')) + 50
    else:
        img_height = max(cell_df['y')) + 50
        img_width = max(cell_df['x')) + 50
    
    # Set axes limits to match image or data
    ax.set_xlim(0, img_width)
    ax.set_ylim(0, img_height)
    
    # Get unique timesteps
    timesteps = sorted(cell_df['timestep'].unique())
    
    # Use only every Nth timestep for animation
    timesteps = timesteps[::frame_interval]
    
    # Create a scatter plot object that will be updated
    scatter = ax.scatter([], [], s=5, color='blue')
    
    # Set title and labels
    title = ax.set_title(f"Cell Position - Model: {model_type}, Sim: {sim_id}\nTimestep: 0", 
                       fontsize=TITLE_FONTSIZE)
    ax.set_xlabel("X Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel("Y Position", fontsize=AXIS_LABEL_FONTSIZE)
    
    # Initialize path points to show the trail
    path_line, = ax.plot([], [], 'r-', alpha=0.5, linewidth=1.5)
    
    # Keep track of centroid positions for path
    centroid_xs, centroid_ys = [], []
    
    def init():
        scatter.set_offsets(np.empty((0, 2)))
        path_line.set_data([], [))
        return scatter, path_line, title
    
    def update(frame_idx):
        if frame_idx >= len(timesteps):
            return scatter, path_line, title
            
        timestep = timesteps[frame_idx]
        
        # Get points for this timestep
        points = cell_df[cell_df['timestep'] == timestep]
        
        if points.empty:
            return scatter, path_line, title
        
        # Update scatter plot with new cell positions
        scatter.set_offsets(np.column_stack((points['x'], points['y'])))
        
        # Calculate and store centroid
        centroid_x = points['x'].mean()
        centroid_y = points['y'].mean()
        centroid_xs.append(centroid_x)
        centroid_ys.append(centroid_y)
        
        # Update path line
        path_line.set_data(centroid_xs, centroid_ys)
        
        # Update title with current timestep
        title.set_text(f"Cell Position - Model: {model_type}, Sim: {sim_id}\nTimestep: {timestep}")
        
        return scatter, path_line, title
    
    # Create animation
    anim = animation.FuncAnimation(fig, update, frames=len(timesteps),
                                 init_func=init, blit=True, interval=1000/fps)
    
    plt.tight_layout()
    
    # Save animation if output path is provided
    if output_path:
        if output_path.endswith('.mp4'):
            writer = animation.FFMpegWriter(fps=fps, bitrate=3600)
            anim.save(output_path, writer=writer)
        elif output_path.endswith('.gif'):
            anim.save(output_path, writer='pillow', fps=fps)
        else:
            anim.save(output_path + '.mp4', writer='ffmpeg', fps=fps)
        print(f"Animation saved to {output_path}")
    
    return anim

def get_domain_tiff_path(folder_path):
    """
    Find the domain TIFF file in the simulation folder.
    
    Args:
        folder_path: Path to the simulation folder
        
    Returns:
        Path to the domain TIFF file, or None if not found
    """
    folder = Path(folder_path)
    
    # Look for domain files with common naming patterns
    domain_patterns = [
        '*_domain*.tif*',
        '*domain_*.tif*',
        '*Domain*.tif*',
        # Add more patterns if needed
    ]
    
    for pattern in domain_patterns:
        matching_files = list(folder.glob(pattern))
        if matching_files:
            return str(matching_files[0))
    
    # If no domain file found, check model.xml for reference
    model_path = folder / 'model.xml'
    if model_path.exists():
        try:
            import xml.etree.ElementTree as ET
            tree = ET.parse(model_path)
            root = tree.getroot()
            
            # Find domain file reference
            image_elem = root.find('.//Space/Lattice/Domain/Image')
            if image_elem is not None and 'path' in image_elem.attrib:
                domain_file = image_elem.attrib['path']
                if domain_file:
                    # Check if this file exists directly in the folder
                    domain_file_path = folder / os.path.basename(domain_file)
                    if domain_file_path.exists():
                        return str(domain_file_path)
                    
                    # Check for similar filenames
                    base_name = os.path.splitext(os.path.basename(domain_file))[0]
                    for ext in ['.tif', '.tiff', '.TIF', '.TIFF']:
                        possible_path = folder / f"{base_name}{ext}"
                        if possible_path.exists():
                            return str(possible_path)
        except Exception as e:
            print(f"Error parsing model.xml to find domain file: {e}")
    
    # Fall back to any TIFF file that might be a domain
    all_tiffs = list(folder.glob('*.tif*'))
    for tiff in all_tiffs:
        if 'domain' in str(tiff).lower():
            return str(tiff)
    
    # Last resort: any TIFF file
    if all_tiffs:
        return str(all_tiffs[0))
    
    return None

def load_domain_image(tiff_path):
    """
    Load a domain TIFF file and convert it to a proper RGB image for display.
    
    Args:
        tiff_path: Path to the TIFF file
        
    Returns:
        RGB image suitable for display
    """
    # Convert Path object to string if needed
    if hasattr(tiff_path, 'exists'):  # It's a Path object
        tiff_path = str(tiff_path)
        
    if not tiff_path or not Path(tiff_path).exists():
        return None
    
    try:
        # Try to load with OpenCV first
        img = cv2.imread(tiff_path)
        
        # If OpenCV fails (e.g., for 16-bit TIFFs), try with PIL/Pillow
        if img is None:
            from PIL import Image
            with Image.open(tiff_path) as pil_img:
                # Convert to numpy array and proper format
                img_array = np.array(pil_img)
                
                # Handle different bit depths and channels
                if len(img_array.shape) == 2:  # Grayscale
                    img = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)
                elif len(img_array.shape) == 3 and img_array.shape[2] == 3:  # RGB
                    img = img_array
                elif len(img_array.shape) == 3 and img_array.shape[2] == 4:  # RGBA
                    img = img_array[:, :, :3]  # Drop alpha channel
                
                # Normalize if 16-bit
                if img.dtype == np.uint16:
                    img = (img / 256).astype(np.uint8)
        else:
            # Convert BGR to RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        return img
    
    except Exception as e:
        print(f"Error loading domain image {tiff_path}: {e}")
        return None

def extract_cell_shape_data_improved(simulation_id: str, 
                                   time_interval: int = 1, 
                                   sampling_density: int = 1000,  # Increased for higher density
                                   interpolate: bool = True,      # Add interpolation
                                   save_path: str = None) -> pd.DataFrame:
    """
    Enhanced version of extract_cell_shape_data with better sampling and continuity.
    
    Args:
        simulation_id: ID of the simulation to process
        time_interval: Process every Nth frame (default=1 means every frame)
        sampling_density: Target number of points per cell (higher = more detailed)
        interpolate: Whether to interpolate between frames to fill gaps
        save_path: Optional path to save the resulting DataFrame
        
    Returns:
        DataFrame with columns: simulation_id, timestep, x, y, frame_index, time_fraction
    """
    # Find the simulation folder path from summary_df
    sim_row = summary_df[summary_df['simulation_id'] == str(simulation_id)]
    if sim_row.empty:
        raise ValueError(f"Simulation ID {simulation_id} not found in summary data")
        
    folder_path = sim_row['folder_path'].iloc[0]
    # Fix the folder path if needed
    folder_path = fix_folder_path(folder_path)
    
    model_type = sim_row['model_type'].iloc[0]
    print(f"Extracting cell shape data for simulation {simulation_id} ({model_type}) from {folder_path}")
    
    # Get the PNG files for this simulation
    png_files = find_simulation_pngs(folder_path)
    
    if len(png_files) == 0:
        raise ValueError(f"No PNG files found for simulation {simulation_id}")
    
    # Get time range for normalization
    min_time = min(t for t, _ in png_files)
    max_time = max(t for t, _ in png_files)
    time_range = max_time - min_time if max_time > min_time else 1
    
    # First pass: Get cell centroids for each frame for potential interpolation
    if interpolate:
        centroids = []
        for timestep, filepath in png_files:
            try:
                image = cv2.imread(filepath)
                if image is None:
                    continue
                black_mask = detect_black_pixels(image, tolerance=10)
                cell_pixel_coords = np.where(black_mask == 255)
                if len(cell_pixel_coords[0)) == 0:
                    continue
                h, w = black_mask.shape
                # Calculate centroid
                centroid_y_img = np.mean(cell_pixel_coords[0))
                centroid_x_img = np.mean(cell_pixel_coords[1))
                # Convert to desired coordinate system (bottom-left = 0,0)
                centroid_y = h - centroid_y_img  # flip y
                centroid_x = centroid_x_img
                centroids.append((timestep, centroid_x, centroid_y))
            except Exception as e:
                print(f"  Error processing centroid for frame {timestep}: {e}")
        centroids.sort(key=lambda x: x[0))  # Sort by timestep
    
    # Process every nth frame based on time_interval
    cell_data = []
    total_frames = len(png_files)
    
    # Progress tracking
    print(f"Processing {len(png_files[::time_interval))} frames out of {total_frames} total frames")
    
    for i, (timestep, filepath) in enumerate(png_files[::time_interval)):
        if i % 10 == 0:
            print(f"  Processing frame {i} (timestep {timestep})")
            
        # Load image and detect black pixels (the cell)
        try:
            image = cv2.imread(filepath)
            if image is None:
                print(f"  Warning: Could not load image {filepath}, skipping")
                continue
                
            # Create binary mask of cell (black) pixels
            black_mask = detect_black_pixels(image, tolerance=10)
            
            # Get coordinates of all cell pixels
            cell_pixel_coords = np.where(black_mask == 255)
            
            # Skip frames where no cell is detected
            if len(cell_pixel_coords[0)) == 0:
                continue
                
            # Convert to rows for DataFrame with image coordinates
            h, w = black_mask.shape
            
            # Calculate normalized time fraction for continuous coloring
            time_fraction = (timestep - min_time) / time_range
            
            # Adaptive sampling based on cell size
            # Use a much higher sampling rate to avoid gaps
            total_cell_pixels = len(cell_pixel_coords[0))
            sampling_rate = max(1, total_cell_pixels // sampling_density)
            
            # Sample cell pixels
            for y_idx, x_idx in zip(cell_pixel_coords[0][::sampling_rate], 
                                    cell_pixel_coords[1][::sampling_rate)):
                # Convert to coordinate system where (0,0) is bottom-left
                y_coord = h - y_idx  # Flip Y coordinate for visualization
                x_coord = x_idx
                
                cell_data.append({
                    'simulation_id': simulation_id,
                    'model_type': model_type,
                    'timestep': timestep,
                    'frame_index': i,
                    'x': x_coord,
                    'y': y_coord,
                    'time_fraction': time_fraction
                })
                
        except Exception as e:
            print(f"  Error processing frame {timestep}: {e}")
    
    # Convert to DataFrame
    if not cell_data:
        raise ValueError(f"No valid cell data extracted from simulation {simulation_id}")
        
    cell_df = pd.DataFrame(cell_data)
    
    # Interpolate between frames if requested
    if interpolate and len(centroids) > 2:
        print("Interpolating between frames to fill gaps...")
        
        # Sort data by timestep to ensure chronological order
        cell_df = cell_df.sort_values('timestep').reset_index(drop=True)
        
        # Get unique timesteps and identify gaps
        timesteps = sorted(cell_df['timestep'].unique())
        
        # Identify large gaps between consecutive timesteps
        gaps = []
        for i in range(len(timesteps) - 1):
            if timesteps[i+1] - timesteps[i] > time_interval * 2:  # If gap is more than 2x the time_interval
                gaps.append((timesteps[i], timesteps[i+1)))
        
        # If we have centroids and gaps, interpolate
        if centroids and gaps:
            print(f"Found {len(gaps)} gaps in the data. Interpolating...")
            interpolated_data = []
            
            # Create a map of timestep -> centroid for easier lookup
            centroid_map = {t: (x, y) for t, x, y in centroids}
            
            # For each gap, create interpolated points
            for start_t, end_t in gaps:
                if start_t in centroid_map and end_t in centroid_map:
                    start_x, start_y = centroid_map[start_t]
                    end_x, end_y = centroid_map[end_t]
                    
                    # Determine number of interpolated frames
                    num_interp = int((end_t - start_t) / time_interval) - 1
                    if num_interp <= 0:
                        continue
                    
                    # Interpolate
                    for i in range(1, num_interp + 1):
                        interp_t = start_t + i * time_interval
                        progress = i / (num_interp + 1)
                        interp_x = start_x + progress * (end_x - start_x)
                        interp_y = start_y + progress * (end_y - start_y)
                        interp_time_fraction = (interp_t - min_time) / time_range
                        
                        # Create multiple points around the interpolated centroid for better coverage
                        # Use a radial pattern to create a circular cell shape
                        radius = 10  # Approximate cell radius in pixels
                        num_points = 30  # Number of points to generate
                        for r in range(3):  # 3 concentric circles
                            for theta in np.linspace(0, 2*np.pi, num_points):
                                x = interp_x + (r+1)*radius/3 * np.cos(theta)
                                y = interp_y + (r+1)*radius/3 * np.sin(theta)
                                
                                interpolated_data.append({
                                    'simulation_id': simulation_id,
                                    'model_type': model_type,
                                    'timestep': interp_t,
                                    'frame_index': -1,  # Mark as interpolated
                                    'x': x,
                                    'y': y,
                                    'time_fraction': interp_time_fraction,
                                    'interpolated': True
                                })
            
            # Add the interpolated data to our dataframe
            if interpolated_data:
                interp_df = pd.DataFrame(interpolated_data)
                cell_df = pd.concat([cell_df, interp_df], ignore_index=True)
                cell_df = cell_df.sort_values('timestep').reset_index(drop=True)
                print(f"Added {len(interpolated_data)} interpolated points")
    
    print(f"Extracted {len(cell_df)} cell pixel positions across {cell_df['timestep'].nunique()} timesteps")
    
    # Save to file if requested
    if save_path:
        cell_df.to_pickle(save_path)
        print(f"Cell shape data saved to {save_path}")
        
    return cell_df

In [ ]:
def filter_timesteps_by_end_region(cell_df, selected_timesteps, domain_width, domain_height, 
                                   end_region_x_max=50, end_region_y_min=400):
    """
    Filter timesteps to stop plotting once the cell centroid reaches near the end of the maze.
    The end of the maze is assumed to be in the top-left corner area.
    
    Args:
        cell_df: DataFrame with cell pixel positions
        selected_timesteps: List of timesteps to potentially filter
        domain_width: Width of the domain
        domain_height: Height of the domain
        end_region_x_max: Maximum x coordinate for end region (left side of maze)
        end_region_y_min: Minimum y coordinate for end region (top of maze, in domain coords where y=0 is bottom)
        
    Returns:
        Filtered list of timesteps (stops once centroid enters end region)
    """
    filtered_timesteps = []
    
    for timestep in selected_timesteps:
        timestep_data = cell_df[cell_df['timestep'] == timestep]
        
        if timestep_data.empty:
            continue
        
        # Calculate centroid for this timestep
        centroid_x = timestep_data['x'].mean()
        centroid_y = timestep_data['y'].mean()
        
        # Check if centroid has reached the end region (top left)
        if centroid_x <= end_region_x_max and centroid_y >= end_region_y_min:
            # Include this timestep (when it first reaches the end) and stop
            filtered_timesteps.append(timestep)
            break
        
        filtered_timesteps.append(timestep)
    
    return filtered_timesteps

def create_cell_mask_from_points(cell_points, domain_width, domain_height, gaussian_sigma=0):
    """
    Create a filled binary mask from cell pixel coordinates.
    Preserves the actual cell shape without using convex hull approximation.
    
    Args:
        cell_points: DataFrame or array with 'x' and 'y' coordinates
        domain_width: Width of the output mask
        domain_height: Height of the output mask
        gaussian_sigma: Standard deviation for Gaussian smoothing (0 = no smoothing)
                       Higher values produce smoother filled shapes
                       Recommended range: 0 (no smoothing) to 5 (heavy smoothing)
            
    Returns:
        Binary mask where cell pixels are True (255)
    """
    # Create empty mask
    mask = np.zeros((int(domain_height), int(domain_width)), dtype=np.uint8)
    
    if isinstance(cell_points, pd.DataFrame):
        x_coords = cell_points['x'].values
        y_coords = cell_points['y'].values
    else:
        x_coords = cell_points[:, 0]
        y_coords = cell_points[:, 1]
    
    if len(x_coords) == 0:
        return mask
    
    # Convert to pixel coordinates (image coordinates have y=0 at top)
    pixel_coords = np.column_stack((
        x_coords,
        domain_height - y_coords  # Flip y for image coordinates
    )).astype(np.int32)
    
    # Remove out-of-bounds points
    valid_mask = (
        (pixel_coords[:, 0] >= 0) & (pixel_coords[:, 0] < domain_width) &
        (pixel_coords[:, 1] >= 0) & (pixel_coords[:, 1] < domain_height)
    )
    pixel_coords = pixel_coords[valid_mask]
    
    if len(pixel_coords) == 0:
        return mask
    
    # Mark all cell pixels directly in the mask (preserves actual shape)
    for px, py in pixel_coords:
        if 0 <= py < domain_height and 0 <= px < domain_width:
            mask[py, px] = 255
    
    # Fill holes in the mask to create a solid shape
    # Use morphological closing to connect nearby pixels
    kernel_size = 3
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    
    # Fill any remaining holes by finding and filling contours
    contours_temp, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours_temp:
        cv2.drawContours(mask, contours_temp, -1, 255, thickness=cv2.FILLED)
    
    # Apply Gaussian smoothing if requested
    if gaussian_sigma > 0:
        # Smooth the binary mask
        mask_float = mask.astype(np.float32) / 255.0
        mask_smooth = cv2.GaussianBlur(mask_float, (0, 0), gaussian_sigma)
        # Threshold back to binary
        mask = (mask_smooth > 0.5).astype(np.uint8) * 255
    
    return mask

def plot_cell_snapshots_filled(cell_df, 
                               background_image_path=None, 
                               output_path=None,
                               timestep_interval=50,
                               alpha=0.5,
                               domain_width=300,
                               domain_height=450,
                               overlay_boundary=True,
                               gaussian_sigma=0,
                               stop_at_end=False,
                               end_region_x_max=50,
                               end_region_y_min=400):
    """
    Plot filled cell shapes at specific timesteps with time-based coloring.
    Uses imshow with masks for smooth, filled appearance instead of scatter dots.
    
    Args:
        cell_df: DataFrame with cell pixel positions
        background_image_path: Optional path to a background maze image
        output_path: Optional path to save the figure
        timestep_interval: Plot every Nth timestep
        alpha: Transparency for cell shapes (0.0 to 1.0)
        domain_width: Width of the domain for scaling
        domain_height: Height of the domain for scaling
        overlay_boundary: If True, overlay the domain boundary on top
        gaussian_sigma: Standard deviation for Gaussian smoothing (0 = no smoothing, 1-5 = light to heavy smoothing)
        stop_at_end: If True, stop plotting once cell centroid reaches the end region
        end_region_x_max: Maximum x coordinate for end region (left side of maze)
        end_region_y_min: Minimum y coordinate for end region (top of maze, in domain coords)
    """
    if cell_df.empty:
        raise ValueError("Empty cell data provided")
    
    # Setup the figure
    fig, ax = plt.subplots(figsize=(10, 15))
    
    # Get simulation info for title
    sim_id = cell_df['simulation_id'].iloc[0]
    model_type = cell_df['model_type'].iloc[0]
    
    # Load background image if provided
    if background_image_path and Path(background_image_path).exists():
        bg_img = load_domain_image(background_image_path)
        if bg_img is not None:
            ax.imshow(bg_img, extent=(0, domain_width, 0, domain_height), alpha=0.3)
    
    # Get unique timesteps and select every Nth one
    all_timesteps = sorted(cell_df['timestep'].unique())
    selected_timesteps = all_timesteps[::timestep_interval]
    
    # Filter timesteps to stop at end region if requested
    if stop_at_end:
        selected_timesteps = filter_timesteps_by_end_region(
            cell_df, selected_timesteps, domain_width, domain_height,
            end_region_x_max, end_region_y_min
        )
    
    if len(selected_timesteps) == 0:
        print("Warning: No timesteps selected with the given interval.")
        return None
    
    print(f"Plotting {len(selected_timesteps)} filled cell snapshots out of {len(all_timesteps)} total timesteps")
    
    # Create colormap for time progression
    colormap = plt.cm.viridis
    
    # Create a composite image for all cell shapes
    composite_mask = np.zeros((int(domain_height), int(domain_width), 4), dtype=np.float32)
    
    # Plot each selected timestep
    for i, timestep in enumerate(selected_timesteps):
        # Get data for this specific timestep
        timestep_data = cell_df[cell_df['timestep'] == timestep]
        
        if timestep_data.empty:
            continue
        
        # Calculate color based on time progression
        if 'time_fraction' in timestep_data.columns:
            time_fraction = timestep_data['time_fraction'].iloc[0]
        else:
            time_fraction = i / (len(selected_timesteps) - 1) if len(selected_timesteps) > 1 else 0
        
        # Get color from colormap (RGBA)
        color = colormap(time_fraction)
        
        # Create mask for this timestep
        mask = create_cell_mask_from_points(timestep_data, domain_width, domain_height, gaussian_sigma)
        
        # Proper alpha compositing using "over" operator
        mask_float = mask.astype(np.float32) / 255.0
        
        # Source layer (new timestep)
        src_alpha = mask_float * alpha
        src_rgb = np.stack([color[0] * src_alpha, color[1] * src_alpha, color[2] * src_alpha], axis=-1)
        
        # Destination layer (existing composite)
        dst_alpha = composite_mask[:, :, 3]
        dst_rgb = composite_mask[:, :, :3]
        
        # Alpha compositing: out_alpha = src_alpha + dst_alpha * (1 - src_alpha)
        out_alpha = src_alpha + dst_alpha * (1 - src_alpha)
        
        # Avoid division by zero
        out_alpha_safe = np.where(out_alpha > 0, out_alpha, 1)
        
        # out_rgb = (src_rgb + dst_rgb * dst_alpha * (1 - src_alpha)) / out_alpha
        out_rgb = (src_rgb + dst_rgb * dst_alpha[..., np.newaxis] * (1 - src_alpha)[..., np.newaxis)) / out_alpha_safe[..., np.newaxis]
        
        # Update composite
        composite_mask[:, :, :3] = out_rgb
        composite_mask[:, :, 3] = out_alpha
    
    # No need to clip anymore - alpha compositing keeps values in [0, 1]
    
    # Display the composite mask
    ax.imshow(composite_mask, extent=(0, domain_width, 0, domain_height), 
             origin='upper', interpolation='bilinear', zorder=5)
    
    # Overlay the domain boundary on top if requested
    if overlay_boundary and background_image_path and Path(background_image_path).exists():
        boundary_mask = extract_domain_boundary(background_image_path, domain_width, domain_height)
        if boundary_mask is not None:
            # Create a black overlay with the boundary
            boundary_overlay = np.zeros((boundary_mask.shape[0], boundary_mask.shape[1], 4), dtype=np.uint8)
            boundary_overlay[boundary_mask > 0] = [0, 0, 0, 255]  # Black with full opacity
            ax.imshow(boundary_overlay, extent=(0, domain_width, 0, domain_height), 
                     interpolation='nearest', zorder=10)
    
    # Create a colorbar to show time progression
    import matplotlib.cm as cm
    from matplotlib.colors import Normalize
    norm = Normalize(vmin=0, vmax=1)
    sm = cm.ScalarMappable(cmap=colormap, norm=norm)
    sm.set_array([))
    cbar = plt.colorbar(sm, ax=ax, label='Time Progression')
    cbar.ax.tick_params(labelsize=TICK_LABEL_FONTSIZE)
    
    # Set plot title and labels
    ax.set_title(f"Cell Snapshots (Filled) - Model: {model_type}, Sim: {sim_id}\n(Every {timestep_interval} timesteps)", 
                fontsize=TITLE_FONTSIZE)
    ax.set_xlabel("X Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel("Y Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.tick_params(axis='both', which='major', labelsize=TICK_LABEL_FONTSIZE)
    
    # Set fixed limits based on domain dimensions
    ax.set_xlim(0, domain_width)
    ax.set_ylim(0, domain_height)
    
    plt.tight_layout()
    
    # Save the figure if output path is specified
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"Filled cell snapshots plot saved to {output_path}")
        plt.close()
    else:
        plt.show()
    
    return fig, ax

def plot_combined_snapshots_filled(simulation_ids, 
                                  background_image_path=None,
                                  output_path=None,
                                  timestep_interval=50,
                                  alpha=0.3,
                                  domain_width=300,
                                  domain_height=450,
                                  overlay_boundary=True,
                                  gaussian_sigma=0,
                                  stop_at_end=False,
                                  end_region_x_max=50,
                                  end_region_y_min=400):
    """
    Plot filled cell shape snapshots for multiple simulations on the same plot.
    Uses imshow with masks for smooth, filled appearance.
    
    Args:
        simulation_ids: Dictionary of model_type -> simulation_id
        background_image_path: Optional path to a background maze image
        output_path: Optional path to save the figure
        timestep_interval: Plot every Nth timestep
        alpha: Transparency for cell shapes (0.0 to 1.0)
        domain_width: Width of the domain for scaling
        domain_height: Height of the domain for scaling
        overlay_boundary: If True, overlay the domain boundary on top
        gaussian_sigma: Standard deviation for Gaussian smoothing (0 = no smoothing, 1-5 = light to heavy smoothing)
        stop_at_end: If True, stop plotting once cell centroid reaches the end region
        end_region_x_max: Maximum x coordinate for end region (left side of maze)
        end_region_y_min: Minimum y coordinate for end region (top of maze, in domain coords)
    """
    # Setup the figure
    fig, ax = plt.subplots(figsize=(10, 15))
    
    # Load background image if provided
    if background_image_path and Path(background_image_path).exists():
        bg_img = load_domain_image(background_image_path)
        if bg_img is not None:
            ax.imshow(bg_img, extent=(0, domain_width, 0, domain_height), alpha=0.3)
    
    # Define distinct colors for each model
    model_colors_hex = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
    from matplotlib.colors import to_rgba
    model_colors = [to_rgba(c) for c in model_colors_hex]
    
    # Create a composite image for all cell shapes
    composite_mask = np.zeros((int(domain_height), int(domain_width), 4), dtype=np.float32)
    
    # Track which models we've added to legend
    legend_added = []
    
    # Plot each model
    for idx, (model, sim_id) in enumerate(simulation_ids.items()):
        # Get cell data
        cell_df = get_or_process_cell_data(sim_id)
        
        if cell_df is None or cell_df.empty:
            print(f"Warning: No data for {model} (sim {sim_id})")
            continue
        
        # Get unique timesteps and select every Nth one
        all_timesteps = sorted(cell_df['timestep'].unique())
        selected_timesteps = all_timesteps[::timestep_interval]
        
        # Filter timesteps to stop at end region if requested
        if stop_at_end:
            selected_timesteps = filter_timesteps_by_end_region(
                cell_df, selected_timesteps, domain_width, domain_height,
                end_region_x_max, end_region_y_min
            )
        
        print(f"Plotting {len(selected_timesteps)} filled snapshots for {model}")
        
        color = model_colors[idx % len(model_colors)]
        
        # Plot each selected timestep for this model
        for timestep in selected_timesteps:
            timestep_data = cell_df[cell_df['timestep'] == timestep]
            
            if timestep_data.empty:
                continue
            
            # Create mask for this timestep
            mask = create_cell_mask_from_points(timestep_data, domain_width, domain_height, gaussian_sigma)
            
            # Proper alpha compositing using "over" operator
            mask_float = mask.astype(np.float32) / 255.0
            
            # Source layer (new timestep)
            src_alpha = mask_float * alpha
            src_rgb = np.stack([color[0] * src_alpha, color[1] * src_alpha, color[2] * src_alpha], axis=-1)
            
            # Destination layer (existing composite)
            dst_alpha = composite_mask[:, :, 3]
            dst_rgb = composite_mask[:, :, :3]
            
            # Alpha compositing: out_alpha = src_alpha + dst_alpha * (1 - src_alpha)
            out_alpha = src_alpha + dst_alpha * (1 - src_alpha)
            
            # Avoid division by zero
            out_alpha_safe = np.where(out_alpha > 0, out_alpha, 1)
            
            # out_rgb = (src_rgb + dst_rgb * dst_alpha * (1 - src_alpha)) / out_alpha
            out_rgb = (src_rgb + dst_rgb * dst_alpha[..., np.newaxis] * (1 - src_alpha)[..., np.newaxis)) / out_alpha_safe[..., np.newaxis]
            
            # Update composite
            composite_mask[:, :, :3] = out_rgb
            composite_mask[:, :, 3] = out_alpha
    
    # No need to clip anymore - alpha compositing keeps values in [0, 1]
    
    # Display the composite mask
    ax.imshow(composite_mask, extent=(0, domain_width, 0, domain_height), 
             origin='upper', interpolation='bilinear', zorder=5)
    
    # Overlay the domain boundary on top if requested
    if overlay_boundary and background_image_path and Path(background_image_path).exists():
        boundary_mask = extract_domain_boundary(background_image_path, domain_width, domain_height)
        if boundary_mask is not None:
            # Create a black overlay with the boundary
            boundary_overlay = np.zeros((boundary_mask.shape[0], boundary_mask.shape[1], 4), dtype=np.uint8)
            boundary_overlay[boundary_mask > 0] = [0, 0, 0, 255]  # Black with full opacity
            ax.imshow(boundary_overlay, extent=(0, domain_width, 0, domain_height), 
                     interpolation='nearest', zorder=10)
    
    # Create manual legend with model colors
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=model_colors_hex[i], label=model) 
                      for i, model in enumerate(simulation_ids.keys())]
    ax.legend(handles=legend_elements, title="Model Types", 
             fontsize=TICK_LABEL_FONTSIZE, loc='upper right')
    
    # Set plot title and labels
    ax.set_title(f"Cell Snapshots (Filled) for Multiple Models\n(Every {timestep_interval} timesteps)", 
                fontsize=TITLE_FONTSIZE)
    ax.set_xlabel("X Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel("Y Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.tick_params(axis='both', which='major', labelsize=TICK_LABEL_FONTSIZE)
    
    # Set fixed limits based on domain dimensions
    ax.set_xlim(0, domain_width)
    ax.set_ylim(0, domain_height)
    
    plt.tight_layout()
    
    # Save the figure if output path is specified
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"Combined filled snapshots plot saved to {output_path}")
        plt.close()
    else:
        plt.show()
    
    return fig, ax

In [ ]:
def extract_cell_perimeter(cell_points, domain_width, domain_height, gaussian_sigma=0):
    """
    Extract the perimeter (boundary) of a cell from its pixel coordinates.
    Preserves the actual cell shape without using convex hull approximation.
    
    Args:
        cell_points: DataFrame or array with 'x' and 'y' coordinates
        domain_width: Width of the domain
        domain_height: Height of the domain
        gaussian_sigma: Standard deviation for Gaussian smoothing (0 = no smoothing)
                       Higher values produce smoother perimeters
                       Recommended range: 0 (no smoothing) to 5 (heavy smoothing)
        
    Returns:
        Array of (x, y) coordinates representing the cell perimeter
    """
    if isinstance(cell_points, pd.DataFrame):
        x_coords = cell_points['x'].values
        y_coords = cell_points['y'].values
    else:
        x_coords = cell_points[:, 0]
        y_coords = cell_points[:, 1]
    
    if len(x_coords) < 3:
        # Not enough points to form a perimeter
        return np.column_stack((x_coords, y_coords))
    
    # Convert to pixel coordinates for contour detection
    pixel_coords = np.column_stack((
        x_coords,
        domain_height - y_coords  # Flip y for image coordinates
    )).astype(np.int32)
    
    # Remove out-of-bounds points
    valid_mask = (
        (pixel_coords[:, 0] >= 0) & (pixel_coords[:, 0] < domain_width) &
        (pixel_coords[:, 1] >= 0) & (pixel_coords[:, 1] < domain_height)
    )
    pixel_coords = pixel_coords[valid_mask]
    
    if len(pixel_coords) < 3:
        # Convert back to domain coordinates
        perimeter_y_domain = domain_height - pixel_coords[:, 1]
        return np.column_stack((pixel_coords[:, 0], perimeter_y_domain))
    
    # Create a binary mask from the cell points by directly marking pixels
    mask = np.zeros((int(domain_height), int(domain_width)), dtype=np.uint8)
    
    # Mark all cell pixels in the mask
    for px, py in pixel_coords:
        if 0 <= py < domain_height and 0 <= px < domain_width:
            mask[py, px] = 255
    
    # Fill holes in the mask to create a solid shape
    # Use morphological closing to connect nearby pixels
    kernel_size = 3
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    
    # Fill any remaining holes
    # Find contours and fill them
    contours_temp, _ = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours_temp:
        cv2.drawContours(mask, contours_temp, -1, 255, thickness=cv2.FILLED)
    
    # Apply Gaussian smoothing if requested
    if gaussian_sigma > 0:
        # Smooth the binary mask
        mask_float = mask.astype(np.float32) / 255.0
        mask_smooth = cv2.GaussianBlur(mask_float, (0, 0), gaussian_sigma)
        # Threshold back to binary
        mask = (mask_smooth > 0.5).astype(np.uint8) * 255
    
    # Find contours to extract the perimeter
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    
    if contours:
        # Get the largest contour (should be the cell)
        largest_contour = max(contours, key=cv2.contourArea)
        
        # Extract perimeter points from contour
        perimeter_pixels = largest_contour.reshape(-1, 2)
        
        # Convert back to domain coordinates (flip y back)
        perimeter_x = perimeter_pixels[:, 0]
        perimeter_y = domain_height - perimeter_pixels[:, 1]
        
        return np.column_stack((perimeter_x, perimeter_y))
    else:
        # Fallback: return the original points
        perimeter_y_domain = domain_height - pixel_coords[:, 1]
        return np.column_stack((pixel_coords[:, 0], perimeter_y_domain))


def plot_perimeter_snapshots(cell_df, 
                             background_image_path=None, 
                             output_path=None,
                             timestep_interval=50,
                             line_width=2,
                             line_alpha=0.7,
                             domain_width=300,
                             domain_height=450,
                             overlay_boundary=True,
                             gaussian_sigma=0):
    """
    Plot cell perimeters (boundaries) at specific timesteps with time-based coloring.
    Instead of showing the full cell shape, this shows only the outer boundary.
    
    Args:
        cell_df: DataFrame with cell pixel positions including 'time_fraction' column
        background_image_path: Optional path to a background maze image
        output_path: Optional path to save the figure
        timestep_interval: Plot every Nth timestep (e.g., 50 means plot at timesteps 0, 50, 100, ...)
        line_width: Width of the perimeter line
        line_alpha: Alpha (transparency) for perimeter lines at each timestep
        domain_width: Width of the domain for scaling
        domain_height: Height of the domain for scaling
        overlay_boundary: If True, overlay the domain boundary on top of the cell perimeters
        gaussian_sigma: Standard deviation for Gaussian smoothing (0 = no smoothing, 1-5 = light to heavy smoothing)
    """
    if cell_df.empty:
        raise ValueError("Empty cell data provided")
    
    # Setup the figure
    fig, ax = plt.subplots(figsize=(10, 15))
    
    # Get simulation info for title
    sim_id = cell_df['simulation_id'].iloc[0]
    model_type = cell_df['model_type'].iloc[0]
    
    # Load background image if provided
    if background_image_path and Path(background_image_path).exists():
        bg_img = load_domain_image(background_image_path)
        if bg_img is not None:
            ax.imshow(bg_img, extent=(0, domain_width, 0, domain_height), alpha=0.3)
    
    # Get unique timesteps and select every Nth one
    all_timesteps = sorted(cell_df['timestep'].unique())
    selected_timesteps = all_timesteps[::timestep_interval]
    
    if len(selected_timesteps) == 0:
        print("Warning: No timesteps selected with the given interval.")
        return None
    
    print(f"Plotting {len(selected_timesteps)} cell perimeter snapshots out of {len(all_timesteps)} total timesteps")
    
    # Create colormap for time progression
    colormap = plt.cm.viridis
    
    # Plot each selected timestep
    for i, timestep in enumerate(selected_timesteps):
        # Get data for this specific timestep
        timestep_data = cell_df[cell_df['timestep'] == timestep]
        
        if timestep_data.empty:
            continue
        
        # Calculate color based on time progression
        if 'time_fraction' in timestep_data.columns:
            time_fraction = timestep_data['time_fraction'].iloc[0]
        else:
            time_fraction = i / (len(selected_timesteps) - 1) if len(selected_timesteps) > 1 else 0
        
        # Get color from colormap
        color = colormap(time_fraction)
        
        # Extract perimeter for this timestep
        perimeter = extract_cell_perimeter(timestep_data, domain_width, domain_height, gaussian_sigma)
        
        if len(perimeter) > 0:
            # Close the perimeter by adding the first point at the end
            perimeter_closed = np.vstack([perimeter, perimeter[0]))
            
            # Plot the perimeter as a line
            ax.plot(perimeter_closed[:, 0], perimeter_closed[:, 1],
                   color=color, linewidth=line_width, alpha=line_alpha, 
                   solid_capstyle='round', solid_joinstyle='round')
    
    # Overlay the domain boundary on top if requested
    if overlay_boundary and background_image_path and Path(background_image_path).exists():
        boundary_mask = extract_domain_boundary(background_image_path, domain_width, domain_height)
        if boundary_mask is not None:
            # Create a black image with the boundary
            boundary_overlay = np.zeros((boundary_mask.shape[0], boundary_mask.shape[1], 4), dtype=np.uint8)
            boundary_overlay[boundary_mask > 0] = [0, 0, 0, 255]  # Black with full opacity
            ax.imshow(boundary_overlay, extent=(0, domain_width, 0, domain_height), 
                     interpolation='nearest', zorder=10)
    
    # Create a colorbar to show time progression
    import matplotlib.cm as cm
    from matplotlib.colors import Normalize
    norm = Normalize(vmin=0, vmax=1)
    sm = cm.ScalarMappable(cmap=colormap, norm=norm)
    sm.set_array([))
    cbar = plt.colorbar(sm, ax=ax, label='Time Progression')
    cbar.ax.tick_params(labelsize=TICK_LABEL_FONTSIZE)
    
    # Set plot title and labels
    ax.set_title(f"Cell Perimeter Snapshots Over Time - Model: {model_type}, Sim: {sim_id}\n(Every {timestep_interval} timesteps)", 
                fontsize=TITLE_FONTSIZE)
    ax.set_xlabel("X Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel("Y Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.tick_params(axis='both', which='major', labelsize=TICK_LABEL_FONTSIZE)
    
    # Set fixed limits based on domain dimensions
    ax.set_xlim(0, domain_width)
    ax.set_ylim(0, domain_height)
    
    plt.tight_layout()
    
    # Save the figure if output path is specified
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"Cell perimeter snapshots plot saved to {output_path}")
        plt.close()
    else:
        plt.show()
    
    return fig, ax


def plot_combined_perimeter_snapshots(simulation_ids, 
                                      background_image_path=None,
                                      output_path=None,
                                      timestep_interval=50,
                                      line_width=2,
                                      line_alpha=0.6,
                                      domain_width=300,
                                      domain_height=450,
                                      overlay_boundary=True,
                                      gaussian_sigma=0):
    """
    Plot cell perimeter snapshots for multiple simulations on the same plot.
    
    Args:
        simulation_ids: Dictionary of model_type -> simulation_id
        background_image_path: Optional path to a background maze image
        output_path: Optional path to save the figure
        timestep_interval: Plot every Nth timestep
        line_width: Width of the perimeter lines
        line_alpha: Alpha (transparency) for perimeter lines
        domain_width: Width of the domain for scaling
        domain_height: Height of the domain for scaling
        overlay_boundary: If True, overlay the domain boundary on top
        gaussian_sigma: Standard deviation for Gaussian smoothing (0 = no smoothing, 1-5 = light to heavy smoothing)
    """
    # Setup the figure
    fig, ax = plt.subplots(figsize=(10, 15))
    
    # Load background image if provided
    if background_image_path and Path(background_image_path).exists():
        bg_img = load_domain_image(background_image_path)
        if bg_img is not None:
            ax.imshow(bg_img, extent=(0, domain_width, 0, domain_height), alpha=0.3)
    
    # Define distinct colors for each model
    model_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
    
    # Plot each model
    for idx, (model, sim_id) in enumerate(simulation_ids.items()):
        # Get cell data
        cell_df = get_or_process_cell_data(sim_id)
        
        if cell_df is None or cell_df.empty:
            print(f"Warning: No data for {model} (sim {sim_id})")
            continue
        
        # Get unique timesteps and select every Nth one
        all_timesteps = sorted(cell_df['timestep'].unique())
        selected_timesteps = all_timesteps[::timestep_interval]
        
        print(f"Plotting {len(selected_timesteps)} perimeter snapshots for {model}")
        
        color = model_colors[idx % len(model_colors)]
        
        # Plot each selected timestep for this model
        for timestep in selected_timesteps:
            timestep_data = cell_df[cell_df['timestep'] == timestep]
            
            if timestep_data.empty:
                continue
            
            # Extract perimeter for this timestep
            perimeter = extract_cell_perimeter(timestep_data, domain_width, domain_height, gaussian_sigma)
            
            if len(perimeter) > 0:
                # Close the perimeter by adding the first point at the end
                perimeter_closed = np.vstack([perimeter, perimeter[0]))
                
                # Plot the perimeter as a line
                ax.plot(perimeter_closed[:, 0], perimeter_closed[:, 1],
                       color=color, linewidth=line_width, alpha=line_alpha,
                       solid_capstyle='round', solid_joinstyle='round',
                       label=model if timestep == selected_timesteps[0] else "")
    
    # Overlay the domain boundary on top if requested
    if overlay_boundary and background_image_path and Path(background_image_path).exists():
        boundary_mask = extract_domain_boundary(background_image_path, domain_width, domain_height)
        if boundary_mask is not None:
            # Create a black overlay with the boundary
            boundary_overlay = np.zeros((boundary_mask.shape[0], boundary_mask.shape[1], 4), dtype=np.uint8)
            boundary_overlay[boundary_mask > 0] = [0, 0, 0, 255]  # Black with full opacity
            ax.imshow(boundary_overlay, extent=(0, domain_width, 0, domain_height), 
                     interpolation='nearest', zorder=10)
    
    # Set plot title and labels
    ax.set_title(f"Cell Perimeter Snapshots for Multiple Models\n(Every {timestep_interval} timesteps)", 
                fontsize=TITLE_FONTSIZE)
    ax.set_xlabel("X Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel("Y Position", fontsize=AXIS_LABEL_FONTSIZE)
    
    # Create legend (filter out empty labels)
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), 
             title="Model Types", fontsize=TICK_LABEL_FONTSIZE, loc='upper right')
    
    ax.tick_params(axis='both', which='major', labelsize=TICK_LABEL_FONTSIZE)
    
    # Set fixed limits based on domain dimensions
    ax.set_xlim(0, domain_width)
    ax.set_ylim(0, domain_height)
    
    plt.tight_layout()
    
    # Save the figure if output path is specified
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"Combined perimeter snapshots plot saved to {output_path}")
        plt.close()
    else:
        plt.show()
    
    return fig, ax


def generate_perimeter_snapshot_figures(model_sim_ids, timestep_interval=50, line_alpha=0.7, line_width=2, gaussian_sigma=0):
    """
    Generate individual cell perimeter snapshot figures for each model.
    This function plots only the cell boundaries (perimeters) rather than the entire filled cell shape.
    
    Args:
        model_sim_ids: Dictionary of model_type -> simulation_id
        timestep_interval: Plot every Nth timestep (default: 50)
        line_alpha: Opacity of perimeter lines (0.0 to 1.0, default: 0.7)
        line_width: Width of perimeter lines (default: 2)
        gaussian_sigma: Standard deviation for Gaussian smoothing (0 = no smoothing, 1-5 = light to heavy smoothing)
    """
    for model, sim_id in model_sim_ids.items():
        print(f"\nGenerating perimeter snapshot visualization for {model} (simulation {sim_id})...")
        
        # Get simulation info
        sim_row = summary_df[summary_df['simulation_id'] == str(sim_id)]
        if sim_row.empty:
            print(f"Warning: Simulation ID {sim_id} not found. Skipping.")
            continue
        
        # Get folder path and domain dimensions
        folder_path = fix_folder_path(sim_row['folder_path'].iloc[0))
        domain_width = int(float(sim_row['domain_width'].iloc[0))) if not pd.isna(sim_row['domain_width'].iloc[0)) else 300
        domain_height = int(float(sim_row['domain_height'].iloc[0))) if not pd.isna(sim_row['domain_height'].iloc[0)) else 450
        
        # Find domain TIFF for background
        domain_tiff = get_domain_tiff_path(folder_path)
        
        # Get cell data
        cell_df = get_or_process_cell_data(sim_id)
        
        if cell_df is not None and not cell_df.empty:
            # Create perimeter snapshot visualization
            data_dir = Path("/Users/edwinhuras/Desktop/USRA_2025/Code/Utilities/CollisionDetection_GapsSlalom")
            output_path = data_dir / f"figure_{model}_sim{sim_id}_perimeter_snapshots.png"
            
            plot_perimeter_snapshots(
                cell_df,
                background_image_path=domain_tiff,
                output_path=output_path,
                timestep_interval=timestep_interval,
                line_width=line_width,
                line_alpha=line_alpha,
                domain_width=domain_width,
                domain_height=domain_height,
                gaussian_sigma=gaussian_sigma
            )
            
            print(f"Saved perimeter snapshot figure to {output_path}")


def generate_combined_perimeter_snapshot_figure(model_sim_ids, timestep_interval=100, line_alpha=0.6, line_width=2, gaussian_sigma=0):
    """
    Generate a combined figure showing cell perimeter snapshots for all models.
    This function plots only the cell boundaries (perimeters) rather than the entire filled cell shape.
    
    Args:
        model_sim_ids: Dictionary of model_type -> simulation_id
        timestep_interval: Plot every Nth timestep (default: 100)
        line_alpha: Opacity of perimeter lines (0.0 to 1.0, default: 0.6)
        line_width: Width of perimeter lines (default: 2)
        gaussian_sigma: Standard deviation for Gaussian smoothing (0 = no smoothing, 1-5 = light to heavy smoothing)
    """
    print("\nGenerating combined perimeter snapshot visualization...")
    
    # Use the first simulation to get domain dimensions
    first_model = list(model_sim_ids.keys())[0]
    first_sim_id = model_sim_ids[first_model]
    sim_row = summary_df[summary_df['simulation_id'] == str(first_sim_id)]
    
    if sim_row.empty:
        print("Error: Cannot get domain dimensions.")
        return
    
    domain_width = int(float(sim_row['domain_width'].iloc[0))) if not pd.isna(sim_row['domain_width'].iloc[0)) else 300
    domain_height = int(float(sim_row['domain_height'].iloc[0))) if not pd.isna(sim_row['domain_height'].iloc[0)) else 450
    
    # Get domain TIFF for background
    folder_path = fix_folder_path(sim_row['folder_path'].iloc[0))
    domain_tiff = get_domain_tiff_path(folder_path)
    
    # Create combined visualization
    data_dir = Path("/Users/edwinhuras/Desktop/USRA_2025/Code/Utilities/CollisionDetection_GapsSlalom")
    output_path = data_dir / "figure_combined_perimeter_snapshots.png"
    
    plot_combined_perimeter_snapshots(
        model_sim_ids,
        background_image_path=domain_tiff,
        output_path=output_path,
        timestep_interval=timestep_interval,
        line_width=line_width,
        line_alpha=line_alpha,
        domain_width=domain_width,
        domain_height=domain_height,
        gaussian_sigma=gaussian_sigma
    )
    
    print(f"Saved combined perimeter snapshot figure to {output_path}")

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pathlib import Path
import cv2
from typing import List, Tuple
import random
from scipy import stats
import matplotlib.animation as animation
import re
import os
import time  # Add time module import

# Font size constants for consistent visualization
TITLE_FONTSIZE = 22
SUBTITLE_FONTSIZE = 18
AXIS_LABEL_FONTSIZE = 16
TICK_LABEL_FONTSIZE = 14

def fix_folder_path(folder_path):
    """
    Corrects simulation folder paths if they don't exist in the original location.
    
    Args:
        folder_path: Original path from the summary dataframe
        
    Returns:
        Corrected path that exists on the filesystem
    """
    path = Path(folder_path)
    
    # If the path exists, no need to fix it
    if path.exists():
        return folder_path
    
    # Paths that start with this prefix need to be corrected
    wrong_prefix = "/Users/edwinhuras/Desktop/USRA_2025/Code/Utilities/Data/GapsSlalom"
    correct_prefix = "/Users/edwinhuras/Desktop/USRA_2025/Code/Utilities/CollisionDetection_GapsSlalom/GapsSlalom"
    
    if wrong_prefix in str(path):
        # Replace the incorrect prefix with the correct one
        corrected_path = str(path).replace(wrong_prefix, correct_prefix)
        corrected_path_obj = Path(corrected_path)
        if corrected_path_obj.exists():
            print(f"Path corrected from:\n  {folder_path}\nto:\n  {corrected_path}")
            return corrected_path
    
    # Alternative approach: use just the folder name with the correct base path
    folder_name = path.name
    alternative_path = Path(correct_prefix) / folder_name
    
    if alternative_path.exists():
        print(f"Path corrected using folder name from:\n  {folder_path}\nto:\n  {alternative_path}")
        return str(alternative_path)
    
    # If we can't find a valid path, return the original but warn
    print(f"⚠️ Warning: Could not find a valid path for: {folder_path}")
    return folder_path

In [10]:


def extract_domain_boundary(domain_image_path, domain_width, domain_height):
    """
    Extract the boundary (black pixels) from the domain image.
    
    Args:
        domain_image_path: Path to the domain TIFF file
        domain_width: Target width for scaling
        domain_height: Target height for scaling
        
    Returns:
        Binary mask where boundaries are white (255) and open space is black (0)
    """
    if not domain_image_path or not Path(domain_image_path).exists():
        return None
    
    # Load the domain image
    domain_img = load_domain_image(domain_image_path)
    if domain_img is None:
        return None
    
    # Convert to grayscale if needed
    if len(domain_img.shape) == 3:
        gray = cv2.cvtColor(domain_img, cv2.COLOR_RGB2GRAY)
    else:
        gray = domain_img
    
    # Create a binary mask where black pixels (boundaries) are white in the mask
    # Threshold to identify black pixels (value < 30 is considered black/boundary)
    _, boundary_mask = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY_INV)
    
    return boundary_mask

def plot_cell_path_continuous_fixed(cell_df, 
                                   background_image_path=None, 
                                   output_path=None,
                                   point_size=1.5,
                                   point_alpha=0.7,
                                   domain_width=300,
                                   domain_height=450,
                                   overlay_boundary=True):
    """
    Plot cell path with continuous color gradient based on time progression.
    
    Args:
        cell_df: DataFrame with cell pixel positions including 'time_fraction' column
        background_image_path: Optional path to a background maze image
        output_path: Optional path to save the figure
        point_size: Size of each point
        point_alpha: Alpha (transparency) for points
        domain_width: Width of the domain for scaling
        domain_height: Height of the domain for scaling
        overlay_boundary: If True, overlay the domain boundary on top of the cell path
    """
    if cell_df.empty:
        raise ValueError("Empty cell data provided")
    
    # Setup the figure
    fig, ax = plt.subplots(figsize=(10, 15))
    
    # Get simulation info for title
    sim_id = cell_df['simulation_id'].iloc[0]
    model_type = cell_df['model_type'].iloc[0]
    
    # Load background image if provided
    bg_img = None
    if background_image_path and Path(background_image_path).exists():
        bg_img = load_domain_image(background_image_path)
        if bg_img is not None:
            ax.imshow(bg_img, extent=(0, domain_width, 0, domain_height), alpha=0.3)
    
    # Use time_fraction for coloring if available, otherwise calculate it
    if 'time_fraction' in cell_df.columns:
        colors = cell_df['time_fraction'].values
    else:
        # Fall back to timestep-based coloring
        min_t = cell_df['timestep'].min()
        max_t = cell_df['timestep'].max()
        time_range = max_t - min_t if max_t > min_t else 1
        colors = (cell_df['timestep'] - min_t) / time_range
    
    # Create scatter plot with continuous color gradient
    scatter = ax.scatter(cell_df['x'], cell_df['y'], 
                        c=colors,
                        cmap='viridis',
                        s=point_size,
                        alpha=point_alpha,
                        vmin=0, vmax=1)
    
    # Overlay the domain boundary on top if requested
    if overlay_boundary and background_image_path and Path(background_image_path).exists():
        boundary_mask = extract_domain_boundary(background_image_path, domain_width, domain_height)
        if boundary_mask is not None:
            # Create a black image with the boundary
            boundary_overlay = np.zeros((boundary_mask.shape[0], boundary_mask.shape[1], 4), dtype=np.uint8)
            boundary_overlay[boundary_mask > 0] = [0, 0, 0, 255]  # Black with full opacity
            ax.imshow(boundary_overlay, extent=(0, domain_width, 0, domain_height), 
                     interpolation='nearest', zorder=10)
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax, label='Time Progression')
    cbar.ax.tick_params(labelsize=TICK_LABEL_FONTSIZE)
    
    # Set plot title and labels
    ax.set_title(f"Cell Path Evolution - Model: {model_type}, Sim: {sim_id}", 
                fontsize=TITLE_FONTSIZE)
    ax.set_xlabel("X Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel("Y Position", fontsize=AXIS_LABEL_FONTSIZE)
    ax.tick_params(axis='both', which='major', labelsize=TICK_LABEL_FONTSIZE)
    
    # Set fixed limits based on domain dimensions
    ax.set_xlim(0, domain_width)
    ax.set_ylim(0, domain_height)
    
    plt.tight_layout()
    
    # Save the figure if output path is specified
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        print(f"Path evolution plot saved to {output_path}")
        plt.close()
    else:
        plt.show()
    
    return fig, ax

In [ ]:
# Helper functions required for cell shape extraction
def analyze_png_colors(filepath: str) -> Tuple[float, float, bool]:
    """
    Analyze the color content of a PNG file to determine if it's the correct type.
    
    Args:
        filepath: Path to the PNG file
        
    Returns:
        Tuple of (white_ratio, green_ratio, is_correct_type)
    """
    try:
        img = cv2.imread(filepath)
        if img is None:
            return 0.0, 0.0, False
            
        # Convert to RGB for analysis
        rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        total_pixels = rgb_img.shape[0] * rgb_img.shape[1]
        
        # Check for white pixels (high RGB values)
        white_mask = np.all(rgb_img > 200, axis=2)  # All channels > 200
        white_ratio = np.sum(white_mask) / total_pixels
        
        # Check for green pixels (high G, low R and B)
        green_mask = (rgb_img[:,:,1] > 100) & (rgb_img[:,:,0] < 100) & (rgb_img[:,:,2] < 100)
        green_ratio = np.sum(green_mask) / total_pixels
        
        # Correct type if it has significant white (>30%) and some green (>1%)
        is_correct = white_ratio > 0.3 and green_ratio > 0.01
        
        return white_ratio, green_ratio, is_correct
        
    except Exception as e:
        print(f"Error analyzing {filepath}: {e}")
        return 0.0, 0.0, False

def find_correct_png_pattern(data_folder: str) -> str:
    """
    Find the correct PNG naming pattern by analyzing color content.
    
    Args:
        data_folder: Path to the directory containing PNG files
        
    Returns:
        The correct pattern prefix (e.g., 'plot_', 'plot-1_', 'plot-2_')
    """
    data_path = Path(data_folder)
    
    if not data_path.exists():
        raise FileNotFoundError(f"Data folder not found: {data_folder}")
    
    # Define possible patterns to check
    patterns = ['plot_', 'plot-1_', 'plot-2_', 'plot-3_']
    
    for pattern in patterns:
        # Look for files matching this pattern
        sample_files = list(data_path.glob(f"{pattern}*.png"))
        
        if len(sample_files) == 0:
            continue
        
        # Analyze only 2 files with this pattern for speed
        correct_count = 0
        total_analyzed = 0
        
        for sample_file in sample_files[:2]:  # Check only 2 samples for faster processing
            white_ratio, green_ratio, is_correct = analyze_png_colors(str(sample_file))
            total_analyzed += 1
            
            if is_correct:
                correct_count += 1
        
        # If at least one sample is correct, this is likely our pattern
        if correct_count > 0:  # At least 1 out of 2 files is correct
            return pattern
    
    # Fallback to default pattern
    return 'plot_'

def find_simulation_pngs(data_folder: str) -> List[Tuple[int, str]]:
    """
    Find all PNG files matching the correct pattern in the given data folder.
    
    Args:
        data_folder: Path to the directory containing simulation PNG files
        
    Returns:
        List of tuples (timestep, filepath) sorted by timestep
    """
    # First, find the correct pattern
    pattern_prefix = find_correct_png_pattern(data_folder)
    
    png_files = []
    data_path = Path(data_folder)
    
    # Create regex pattern for extracting timestep number
    pattern = re.compile(rf'{re.escape(pattern_prefix)}(\d+)\.png$')
    
    for file_path in data_path.glob(f"{pattern_prefix}*.png"):
        match = pattern.match(file_path.name)
        if match:
            timestep = int(match.group(1))
            png_files.append((timestep, str(file_path)))
    
    # Sort by timestep for sequential processing
    png_files.sort(key=lambda x: x[0])
    
    return png_files

def detect_black_pixels(image: np.ndarray, tolerance=10, border_exclude=1) -> np.ndarray:
    """
    Create a mask for black pixels representing the cell.
    
    Args:
        image: BGR image
        tolerance: Color tolerance for matching black
        border_exclude: Number of pixels to exclude from each edge
        
    Returns:
        Binary mask where matching black pixels are white (255)
    """
    # Define target color as black (0,0,0)
    target_bgr = (0, 0, 0)  # (B, G, R)
    
    # Create lower and upper bounds with tolerance
    lower_bound = np.array([max(0, target_bgr[0] - tolerance),
                           max(0, target_bgr[1] - tolerance),
                           max(0, target_bgr[2] - tolerance)])
    upper_bound = np.array([min(255, target_bgr[0] + tolerance),
                           min(255, target_bgr[1] + tolerance),
                           min(255, target_bgr[2] + tolerance)])
    
    # Create mask
    black_mask = cv2.inRange(image, lower_bound, upper_bound)
    
    # Exclude border pixels by setting them to 0 (black) in the mask
    if border_exclude > 0:
        h, w = black_mask.shape
        black_mask[:border_exclude, :] = 0  # Top edge
        black_mask[-border_exclude:, :] = 0  # Bottom edge
        black_mask[:, :border_exclude] = 0  # Left edge
        black_mask[:, -border_exclude:] = 0  # Right edge
    
    return black_mask



In [ ]:
def get_maze_background(sim_data):
    """
    Gets a clean maze background image.
    It assumes the first frame of the simulation shows the cell at the start,
    and the maze structure is visible.
    """
    if sim_data.empty or 'image_path' not in sim_data.columns:
        return None
    
    first_frame_path = sim_data['image_path'].iloc[0]
    if not Path(first_frame_path).exists():
        print(f"Warning: Image path not found: {first_frame_path}")
        return None
        
    # Load the first frame, which should contain the maze.
    # This is a proxy for a clean maze image.
    maze_img = cv2.imread(first_frame_path)
    return cv2.cvtColor(maze_img, cv2.COLOR_BGR2RGB)


In [8]:
def rescale_cell_data_to_domain(cell_df, domain_width, domain_height, maze_progress_percentage=100.0):
    """
    Rescale cell coordinates to match the domain dimensions, accounting for maze progress.
    Uses the MAXIMUM centroid Y position ever reached (not just final position) to handle
    cases where the cell bounces around after reaching its maximum progress.
    
    Args:
        cell_df: DataFrame with cell pixel coordinates
        domain_width: Target domain width
        domain_height: Target domain height
        maze_progress_percentage: How far through the maze the cell got (0-100)
        
    Returns:
        DataFrame with rescaled coordinates
    """
    if cell_df is None or cell_df.empty:
        return cell_df
    
    # Calculate centroid Y position for each timestep to find the maximum progress
    # This handles cases where the cell bounces back after reaching max progress
    timestep_centroids = cell_df.groupby('timestep')['y'].mean()
    max_centroid_y = timestep_centroids.max()
    
    # For x-dimension, use the full width (assuming cell explores full width)
    x_max = cell_df['x'].max()
    
    # Calculate the target centroid height based on maze progress
    # If the cell made it 70% through the maze, its centroid should be at 70% of domain_height
    target_centroid_height = domain_height * (maze_progress_percentage / 100.0)
    
    # Calculate scaling factors
    x_scale = domain_width / x_max if x_max > 0 else 1
    y_scale = target_centroid_height / max_centroid_y if max_centroid_y > 0 else 1
    
    # Apply scaling
    cell_df = cell_df.copy()  # Avoid modifying the original
    cell_df['x'] = cell_df['x'] * x_scale
    cell_df['y'] = cell_df['y'] * y_scale
    
    return cell_df

In [9]:
# Persistent cell data cache manager
import time  # Add time module import

def get_or_process_cell_data(simulation_id, force_reprocess=False, sampling_density=2000, time_interval=1):
    """
    Get cell data for a simulation from cache if available, or process it if not.
    
    Args:
        simulation_id: ID of the simulation to get data for
        force_reprocess: If True, reprocess even if cached
        sampling_density: Density parameter for extraction if processing is needed
        time_interval: Process every Nth frame (default=1)
        
    Returns:
        DataFrame with cell data for the requested simulation (rescaled to domain)
    """
    # Path to the persistent cell data cache file
    data_dir = Path("/Users/edwinhuras/Desktop/USRA_2025/Code/Utilities/CollisionDetection_GapsSlalom")
    cache_path = data_dir / 'processed_cell_data_cache.pkl'
    
    # Get domain dimensions and maze progress for scaling
    sim_row = summary_df[summary_df['simulation_id'] == str(simulation_id)]
    domain_width = int(float(sim_row['domain_width'].iloc[0])) if not pd.isna(sim_row['domain_width'].iloc[0]) else 300
    domain_height = int(float(sim_row['domain_height'].iloc[0])) if not pd.isna(sim_row['domain_height'].iloc[0]) else 450
    maze_progress = float(sim_row['maze_progress_percentage'].iloc[0]) if not pd.isna(sim_row['maze_progress_percentage'].iloc[0]) else 100.0
    
    # Try to load the cache if it exists
    cell_data_cache = None
    if cache_path.exists() and not force_reprocess:
        try:
            cell_data_cache = pd.read_pickle(cache_path)
            print(f"Loaded cell data cache with {len(cell_data_cache)} points from {cell_data_cache['simulation_id'].nunique()} simulations")
        except Exception as e:
            print(f"Warning: Could not load cell data cache: {e}")
            cell_data_cache = pd.DataFrame()
    else:
        print("No existing cache found or force_reprocess=True, creating new cache")
        cell_data_cache = pd.DataFrame()
    
    # Check if simulation is already in cache
    # Only check if the cache is not empty and has the simulation_id column
    if not cell_data_cache.empty and 'simulation_id' in cell_data_cache.columns and not force_reprocess:
        sim_data = cell_data_cache[cell_data_cache['simulation_id'] == str(simulation_id)]
        if not sim_data.empty:
            print(f"Simulation {simulation_id} data found in cache, loading...")
            
            # IMPORTANT: Cache stores UNSCALED data, so we apply scaling here
            # Return rescaled data with maze progress taken into account
            return rescale_cell_data_to_domain(sim_data.copy(), domain_width, domain_height, maze_progress)
    
    # If not in cache, process the data using extract_cell_shape_data_improved
    print(f"Processing simulation {simulation_id}...")
    cell_df = extract_cell_shape_data_improved(
        simulation_id, 
        time_interval=time_interval,
        sampling_density=sampling_density,
        interpolate=True,
        save_path=None  # No need to save individual files now
    )
    
    if cell_df is not None and not cell_df.empty:
        # IMPORTANT: Save UNSCALED data to cache to avoid double-scaling issues
        # Add to cache (unscaled)
        if cell_data_cache.empty:
            # If cache is empty, just use the new data as the cache
            cell_data_cache = cell_df.copy()
        else:
            # Remove any existing data for this sim_id to avoid duplicates
            if 'simulation_id' in cell_data_cache.columns:
                cell_data_cache = cell_data_cache[cell_data_cache['simulation_id'] != str(simulation_id)]
            # Add new data
            cell_data_cache = pd.concat([cell_data_cache, cell_df.copy()], ignore_index=True)
        
        # Save updated cache (with unscaled data)
        try:
            cell_data_cache.to_pickle(cache_path)
            print(f"Updated cell data cache with {len(cell_df)} points for simulation {simulation_id}")
            print(f"Total cached simulations: {cell_data_cache['simulation_id'].nunique()}")
        except Exception as e:
            print(f"Warning: Could not save cell data cache: {e}")
        
        # Return rescaled version for immediate use
        return rescale_cell_data_to_domain(cell_df, domain_width, domain_height, maze_progress)
    
    return cell_df

## Generate Visualizations

Generate overlay visualizations for high-progress simulations.

In [ ]:
# Select high-progress simulations for visualization
top_sims = summary_df[summary_df["track_progress_percentage"] > 50].groupby("model_type").first()
model_sim_ids = top_sims["simulation_id"].to_dict()
print("Selected simulations to visualize:", model_sim_ids)
